# 0.6 · Plotly & 交互可视化 / Interactive Visualization

> **课程定位 / Where this fits**
> 第 6 课，**Part 0 · 基础准备**。
> Lesson 6, **Part 0 · Foundations**.
>
> Matplotlib/Seaborn 是"打印出来贴在墙上"的静态图。**Plotly 是"挂到网页上的"交互图**——可悬停、可缩放、可拖动、可动画。**做 Dashboard、给 stakeholder 演示几乎都用它**。
> Matplotlib/Seaborn = "print-and-pin" static plots. **Plotly = interactive web plots** — hover, zoom, pan, animate. **The standard for dashboards and stakeholder demos.**

> 📐 **符号约定 / Notation**（见 [`NOTATION.md`](../NOTATION.md)）
> 本节主要是 API 使用，数学很少。
> Mostly API; minimal math.

> 💡 **何时该用 Plotly / When to pick Plotly**
> - 给非技术 stakeholder 看数据 → **Plotly**（可悬停看具体值）
> - 在 Streamlit / Dash dashboard 里嵌入 → **Plotly**
> - GitHub README / 论文 / Slack → **matplotlib**（静态、易嵌入）
> - 万级以上数据点散点 → **matplotlib**（plotly 卡顿）
>
> **DS 工作里两套都得会**，Plotly 越来越多面试官会问你"怎么把这个图做成可交互的"。
> You need both — Plotly questions show up more and more in interviews ("how would you make this interactive?").

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 区分 **Plotly Express (`px`)** 和 **Graph Objects (`go`)** 两个 API 层级，知道何时用哪个。
   Tell apart **Plotly Express (`px`)** and **Graph Objects (`go`)**, and pick the right level.
2. 一行画出带悬停的散点 / 折线 / 柱状 / 直方图 / 箱线 / 小提琴 / 热图。
   One-liner to draw interactive scatter / line / bar / hist / box / violin / heatmap.
3. **自定义 hover 卡片、配色、模板**（dark / white / presentation）。
   Customize **hover cards, palettes, themes**.
4. 用 **`animation_frame`** 做出"年份滑块" 的动态图（gapminder 著名演示）。
   Make Hans Rosling-style animated bubble charts with **`animation_frame`**.
5. 画 **3D 散点** 和 **地图（choropleth）**。
   3D scatter and **choropleth maps**.
6. 用 `make_subplots` 把多个 Plotly 图拼到一张大画布。
   Combine multiple plots with `make_subplots`.
7. 把交互图导出成 **独立 HTML 文件**，能直接拖给同事看。
   Export to **standalone HTML** to share.

---

## 目录 / Table of Contents

1. [Plotly 的两层 API / Two-level API](#1)
2. [🌍 数据集介绍：Gapminder](#2)
3. [Plotly Express 快速入门 / Quick Start](#3)
4. [常见图表 / Core Plot Types](#4)
5. [Hover 卡片定制 / Customizing Hover](#5)
6. [主题与配色 / Themes & Palettes](#6)
7. [分面 / Faceting](#7)
8. [**动画 / Animation** ⭐](#8)
9. [3D 图表 / 3D Plots](#9)
10. [地图 / Maps](#10)
11. [`make_subplots` 多子图](#11)
12. [Graph Objects 精细控制 / Fine Control with `go`](#12)
13. [保存为 HTML / Saving as HTML](#13)
14. [从图到 Dashboard / From Plot to Dashboard](#14)
15. [实战：Gapminder 完整探索 / Hands-on](#15)
16. [小结 / Summary](#16)


<a id="1"></a>
## 1. Plotly 的两层 API / Two-level API

Plotly 提供**两套 API**，一定要分清：
Plotly has **two APIs** — keep them straight:

| 层级 / Level | 模块 / Module | 写法 / Style | 何时用 / When |
|---|---|---|---|
| **High-level** | `plotly.express` (alias `px`) | **一行调用**：`px.scatter(df, x=..., y=...)` | 95% 的场景 / most cases |
| **Low-level** | `plotly.graph_objects` (alias `go`) | 构建 trace 字典 `go.Scatter(...)` | 精细控制 / fine-grained tweaks |

> **学习策略 / Strategy**：先把 `px` 用熟，遇到 `px` 做不到的事再下沉到 `go`。
> Master `px` first; drop to `go` only when you must.


In [ ]:
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

print(f"plotly: {plotly.__version__}")

# 默认模板 / Default template
pio.templates.default = "plotly_white"


<a id="2"></a>
## 2. 🌍 数据集介绍：Gapminder

> **来源 / Source**: Hans Rosling 的 Gapminder 项目；Plotly 内置 `px.data.gapminder()`。
> Hans Rosling's Gapminder project; bundled in `px.data.gapminder()`.
>
> **背景 / Background**: Hans Rosling 是瑞典统计学家，他用一张著名的**动态气泡图**（GDP × 寿命 × 年份）颠覆了人们对"发展中国家"的刻板印象。**这是数据可视化史上最有名的一张图**。
> Hans Rosling, a Swedish statistician, used his famous animated bubble chart (GDP × life-expectancy × year) to upend stereotypes about developing nations. **The most famous plot in data-viz history.**
>
> **内容 / Contents**: 142 个国家、12 个时间点（1952–2007，每 5 年）= 1704 行。
> 142 countries × 12 time points (1952-2007, every 5 years) = 1704 rows.
>
> | 列 / Column | 含义 / Meaning |
> |---|---|
> | `country`   | 国家名 / country name |
> | `continent` | 大洲 / continent |
> | `year`      | 年份 / year |
> | `lifeExp`   | 出生预期寿命（岁）/ life expectancy at birth |
> | `pop`       | 人口 / population |
> | `gdpPercap` | 人均 GDP（美元）/ GDP per capita ($) |
> | `iso_alpha` | ISO 3 字母国家代码 / ISO alpha-3 code |
> | `iso_num`   | ISO 数字代码 / ISO numeric code |
>
> **为什么经典 / Why classic**: 时间维度 + 多变量 + 地理标签 —— **Plotly 所有 sweet spot 都能秀**：动画、气泡、地图。
> Time dimension + multi-variable + geo tags — perfect for showcasing Plotly's strengths.


In [ ]:
gap = px.data.gapminder()
print(f"shape : {gap.shape}")
print(f"years : {gap['year'].min()} → {gap['year'].max()}")
print(f"countries: {gap['country'].nunique()}")
print(f"continents: {gap['continent'].unique().tolist()}")
gap.head()


<a id="3"></a>
## 3. Plotly Express 快速入门 / Quick Start

**核心套路**：`px.<plot_type>(df, x=..., y=..., color=..., size=..., hover_data=...)`
**Pattern**: pass DataFrame, name columns by string.


In [ ]:
# 散点图 / Scatter — 用 2007 年截面 / 2007 cross-section
gap_2007 = gap[gap["year"] == 2007]

fig = px.scatter(
    gap_2007,
    x="gdpPercap",
    y="lifeExp",
    color="continent",
    size="pop",
    size_max=60,                # 气泡最大像素 / max bubble pixels
    log_x=True,                 # GDP 跨 3 个量级 → 对数 / 3 OOM range → log
    hover_name="country",       # 悬停时的大标题 / hover title
    title="GDP per capita vs Life Expectancy (2007)",
)
fig.show()


**☝ 把鼠标放到任意一个气泡上**，你会看到：国家名、GDP、寿命、人口、大洲——全是默认 hover 包含的信息。
**Hover over any bubble** — country, GDP, life-expectancy, population, continent all appear automatically.

这是 Plotly 的核心价值：**信息密度比静态图大几倍**，因为大部分信息藏在 hover / zoom 里。
This is Plotly's core value: **info density is far higher**, because most details hide in hover / zoom.


<a id="4"></a>
## 4. 常见图表 / Core Plot Types

| 静态对应 / Static counterpart | Plotly Express |
|---|---|
| `plt.plot` / `sns.lineplot` | `px.line` |
| `plt.scatter` / `sns.scatterplot` | `px.scatter` |
| `plt.bar` / `sns.barplot` | `px.bar` |
| `plt.hist` / `sns.histplot` | `px.histogram` |
| `sns.boxplot` | `px.box` |
| `sns.violinplot` | `px.violin` |
| `sns.heatmap` | `px.imshow` |
| `sns.pairplot` | `px.scatter_matrix` |


In [ ]:
# 折线图：中国 vs 印度 寿命变化 / China vs India life-expectancy over time
sub = gap[gap["country"].isin(["China", "India", "United States", "Japan"])]

fig = px.line(
    sub, x="year", y="lifeExp", color="country",
    markers=True,
    title="Life Expectancy over Time",
    labels={"lifeExp": "Life Expectancy (years)", "year": "Year"},
)
fig.show()


In [ ]:
# 柱状图：2007 年各大洲平均寿命 / Bar: mean lifeExp by continent in 2007
by_cont = (
    gap_2007.groupby("continent", as_index=False)["lifeExp"]
    .mean().round(1).sort_values("lifeExp", ascending=False)
)

fig = px.bar(
    by_cont, x="continent", y="lifeExp",
    color="continent",
    text="lifeExp",                            # 在柱顶标数值 / value labels
    title="Mean Life Expectancy by Continent (2007)",
)
fig.update_traces(textposition="outside")
fig.show()


In [ ]:
# 直方图 / Histogram —— 2007 年寿命分布
fig = px.histogram(
    gap_2007, x="lifeExp",
    color="continent",
    nbins=30,
    marginal="box",         # 边缘箱线图！/ marginal box plot — really cool
    title="Life Expectancy distribution in 2007 (with marginal box)",
)
fig.show()


> 💡 **`marginal="box"` / `marginal="rug"` / `marginal="violin"`** 是 Plotly 独特的小心思——在主图边缘自动加一个统计图。Matplotlib/Seaborn 要写好几行才能做到。
> The `marginal=...` argument is a Plotly trick — auto-adds a small stats plot at the margin. Hard to match in static plotting libs.


In [ ]:
# 箱线 / 小提琴 ——按大洲看寿命分布
fig = px.box(
    gap_2007, x="continent", y="lifeExp",
    color="continent",
    points="all",                    # 把每个数据点也画出来 / show all points
    title="Life Expectancy by Continent (2007)",
)
fig.show()


<a id="5"></a>
## 5. Hover 卡片定制 / Customizing Hover

**Plotly 的 hover 信息就是它的核心竞争力**。两种定制方式：
**Hover info is Plotly's killer feature.** Two ways to customize:

1. `hover_data={...}`（声明式，最常用）
2. `hovertemplate=...`（自由文本模板，最强）


In [ ]:
# 1) hover_data：声明式 / Declarative
fig = px.scatter(
    gap_2007,
    x="gdpPercap", y="lifeExp",
    color="continent", log_x=True,
    hover_name="country",
    hover_data={
        "gdpPercap": ":.0f",      # 整数格式 / int format
        "lifeExp": ":.1f",         # 1 位小数 / 1 decimal
        "pop": ":,.0f",            # 千分位 / thousand separator
        "continent": False,        # 不在 hover 显示 / hide from hover
    },
    title="Hover format: gdpPercap, lifeExp, pop have custom formats",
)
fig.show()


In [ ]:
# 2) hovertemplate：自由格式 / Free-text template (most powerful)
fig = px.scatter(
    gap_2007.head(50),
    x="gdpPercap", y="lifeExp", log_x=True,
)
fig.update_traces(
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"           # 大标题：国家名
        "💰 GDP / capita: <b>$%{x:,.0f}</b><br>"
        "🧬 Life expectancy: <b>%{y:.1f} years</b><br>"
        "🌍 Continent: %{customdata[1]}<extra></extra>"
    ),
    customdata=gap_2007.head(50)[["country", "continent"]].values,
)
fig.show()


> **`<extra></extra>`** 把右上角那个小标签干掉（不然 hover 卡片右边会有 "trace 0" 之类的没用信息）。
> The `<extra></extra>` tag removes the unhelpful "trace 0" badge from the hover card.


<a id="6"></a>
## 6. 主题与配色 / Themes & Palettes

Plotly 自带十几个**主题模板**：
Plotly ships built-in themes:

| 模板 / Template | 用途 / Use |
|---|---|
| `plotly_white` | 白底，最干净 / clean white |
| `plotly_dark`  | 暗色 / dark mode |
| `ggplot2`      | 模仿 R ggplot2 |
| `seaborn`      | 模仿 seaborn |
| `simple_white` | 极简 / minimalist |
| `presentation` | 字号大，适合 PPT / big fonts for slides |


In [ ]:
# 同一个图，三种风格 / Same plot, three themes
import plotly.subplots as sp

for tpl in ["plotly_white", "plotly_dark", "presentation"]:
    fig = px.scatter(
        gap_2007, x="gdpPercap", y="lifeExp",
        color="continent", size="pop", size_max=40, log_x=True,
        title=f"Template: {tpl}",
        template=tpl,
    )
    fig.update_layout(height=350)
    fig.show()


In [ ]:
# 离散色板 / Discrete color palettes
print("常用色板 / Popular discrete palettes:")
for name in ["Plotly", "Set1", "Set2", "D3", "Pastel", "Bold", "Vivid"]:
    pal = getattr(px.colors.qualitative, name)
    print(f"  {name:<10}: {pal[:5]}...")

# 连续色板 / Continuous (sequential / diverging) palettes
print("\n连续色板 / Popular continuous palettes:")
print("  Viridis, Plasma, Inferno, Magma  (perceptually uniform / 感知均匀)")
print("  RdBu, Spectral, PiYG              (diverging / 双向)")


In [ ]:
# 应用自定义色板 / Apply custom palette
fig = px.scatter(
    gap_2007, x="gdpPercap", y="lifeExp",
    color="continent", size="pop", size_max=40, log_x=True,
    color_discrete_sequence=px.colors.qualitative.Set2,    # 类别 / discrete
    title="With Set2 palette",
)
fig.show()


<a id="7"></a>
## 7. 分面 / Faceting

和 seaborn 的 `FacetGrid` 类似——按变量分成多个子图。
Like seaborn's `FacetGrid` — split into subplots by variable.


In [ ]:
# 行/列分面 / Row & col facets
fig = px.scatter(
    gap[gap["year"].isin([1952, 1977, 2007])],
    x="gdpPercap", y="lifeExp",
    color="continent", size="pop", size_max=40, log_x=True,
    facet_col="year",                # 横向分 / split horizontally
    title="Gapminder snapshots: 1952 vs 1977 vs 2007",
    height=400,
)
fig.show()


<a id="8"></a>
## 8. 动画 / Animation ⭐ —— Plotly 的"杀手锏"

**Hans Rosling 的著名演示就用这一行复刻**：
**Hans Rosling's famous bubble animation, in one line:**

```python
px.scatter(gap, ..., animation_frame="year", animation_group="country", ...)
```

- `animation_frame=` 告诉 Plotly "按这个列做动画帧"
- `animation_group=` 让 Plotly 知道"同一个东西的不同帧"（这样 transition 才是平滑移动）


In [ ]:
# 著名的 Hans Rosling 动画 / The famous Hans Rosling animation
fig = px.scatter(
    gap,
    x="gdpPercap", y="lifeExp",
    color="continent",
    size="pop", size_max=60,
    hover_name="country",
    log_x=True,
    range_x=[100, 100_000],          # 固定坐标轴范围 / freeze axes for clean animation
    range_y=[25, 90],
    animation_frame="year",
    animation_group="country",
    title="Gapminder: World development 1952–2007",
    height=550,
)
fig.show()


**☝ 按下底部的 ▶ Play 键**，你会看到：
- 1952 年：所有国家都挤在左下角（穷且寿命短）
- 1992 年：东亚国家（红色 Asia）**显著右上漂移**——经济起飞 + 寿命跃升
- 2007 年：绝大多数国家进入 60+ 岁的"高寿命圈"
- 整个 World "right-and-up" 趋势——**世界并不像新闻里那么糟**（Rosling 的核心信息）

这一行代码替代了 Rosling 团队当年几个月的工程量。
This one line replaces months of engineering effort that Rosling's team did originally.


<a id="9"></a>
## 9. 3D 图表 / 3D Plots

3D 散点、3D 表面——拖动旋转。
3D scatter, 3D surface — drag to rotate.


In [ ]:
# 3D 散点 / 3D scatter
fig = px.scatter_3d(
    gap_2007,
    x="gdpPercap", y="pop", z="lifeExp",
    color="continent",
    size="pop", size_max=30,
    log_x=True, log_y=True,         # 两轴都对数 / log both
    hover_name="country",
    title="3D: GDP × Population × Life Expectancy (2007)",
    height=600,
)
fig.show()


> ⚠ **3D 图慎用 / Use 3D sparingly**：
> 3D 看起来酷，但**遮挡严重**，远近难判断。**只有真的有 3 维结构**才用——二维 + 颜色/大小通常更好。
> 3D looks cool but suffers from occlusion + bad depth perception. Use only when there's genuine 3-D structure — 2D + color/size usually beats it.


In [ ]:
# 3D 表面 / 3D surface — 数学函数可视化 / visualize a math function
x = np.linspace(-3, 3, 50)
y = np.linspace(-3, 3, 50)
X, Y = np.meshgrid(x, y)
Z = np.sin(np.sqrt(X**2 + Y**2)) / (np.sqrt(X**2 + Y**2) + 0.1)

fig = go.Figure(data=[go.Surface(z=Z, x=x, y=y, colorscale="Viridis")])
fig.update_layout(
    title="3D surface: sin(r)/r",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="f(x,y)"),
    height=500,
)
fig.show()


<a id="10"></a>
## 10. 地图 / Maps

地理数据是 plotly 的传统强项。两种最常用：
Geo data is Plotly's traditional strength. Two common types:

- `px.choropleth` —— 按国家/州**填色**的"色块图"
- `px.scatter_geo` —— 在地图上**画点**


In [ ]:
# 世界人均 GDP 色块图（2007）/ World GDP-per-capita choropleth (2007)
fig = px.choropleth(
    gap_2007,
    locations="iso_alpha",                # ISO-3 字母代码 / ISO alpha-3
    color="gdpPercap",
    hover_name="country",
    color_continuous_scale="Viridis",
    range_color=(0, 50_000),
    projection="natural earth",
    title="World GDP per capita (2007)",
    height=500,
)
fig.show()


In [ ]:
# 配合 animation_frame 做"每 5 年一帧"的世界寿命变化
# Combined with animation_frame: world life-expectancy over time
fig = px.choropleth(
    gap,
    locations="iso_alpha",
    color="lifeExp",
    hover_name="country",
    animation_frame="year",
    color_continuous_scale="RdYlGn",      # 红→黄→绿，直观 / red-yellow-green
    range_color=(30, 85),
    projection="natural earth",
    title="World life expectancy 1952–2007 (press ▶ play)",
    height=500,
)
fig.show()


<a id="11"></a>
## 11. `make_subplots` 多子图

Plotly Express 不支持任意子图布局——要拼图就用 `make_subplots`。
Plotly Express doesn't support arbitrary subplot layouts — use `make_subplots`.


In [ ]:
# 2x2 layout combining different chart types
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Scatter: GDP vs LifeExp (2007)",
        "Line: 4 countries over time",
        "Bar: top 10 countries by GDP",
        "Box: lifeExp by continent",
    ],
    specs=[[{"type": "xy"}, {"type": "xy"}],
           [{"type": "xy"}, {"type": "xy"}]],
)

# 1) Scatter
fig.add_trace(
    go.Scatter(
        x=gap_2007["gdpPercap"], y=gap_2007["lifeExp"],
        mode="markers", marker=dict(size=8, opacity=0.6),
        text=gap_2007["country"], name="2007",
    ),
    row=1, col=1,
)
fig.update_xaxes(type="log", row=1, col=1)

# 2) Line
for ctry in ["China", "India", "United States", "Japan"]:
    sub = gap[gap["country"] == ctry]
    fig.add_trace(
        go.Scatter(x=sub["year"], y=sub["lifeExp"], mode="lines+markers", name=ctry),
        row=1, col=2,
    )

# 3) Bar
top10 = gap_2007.nlargest(10, "gdpPercap")
fig.add_trace(
    go.Bar(x=top10["country"], y=top10["gdpPercap"], name="GDP"),
    row=2, col=1,
)

# 4) Box
for cont in gap_2007["continent"].unique():
    sub = gap_2007[gap_2007["continent"] == cont]
    fig.add_trace(
        go.Box(y=sub["lifeExp"], name=cont),
        row=2, col=2,
    )

fig.update_layout(height=700, title="4-panel composite", showlegend=False)
fig.show()


<a id="12"></a>
## 12. Graph Objects 精细控制 / Fine Control with `go`

当 `px` 不够用时，下沉到 `go`：手动构建 trace。
When `px` runs out of knobs, drop to `go` and build traces by hand.


In [ ]:
# 手动叠加两个 trace + 复杂 layout
fig = go.Figure()

# Trace 1: 中国寿命
china = gap[gap["country"] == "China"]
fig.add_trace(go.Scatter(
    x=china["year"], y=china["lifeExp"],
    mode="lines+markers",
    name="China life expectancy",
    line=dict(color="crimson", width=3),
    marker=dict(size=10),
))

# Trace 2: 中国 GDP（次坐标轴 / secondary y-axis）
fig.add_trace(go.Scatter(
    x=china["year"], y=china["gdpPercap"],
    mode="lines+markers",
    name="China GDP per capita",
    line=dict(color="steelblue", width=3, dash="dash"),
    yaxis="y2",                       # 用第二个 Y 轴 / use 2nd y-axis
))

fig.update_layout(
    title="China: Life Expectancy & GDP/capita, 1952–2007",
    xaxis=dict(title="Year"),
    yaxis=dict(title="Life Expectancy (years)", side="left"),
    yaxis2=dict(title="GDP / capita ($)", side="right", overlaying="y"),
    legend=dict(x=0.02, y=0.98),
    height=450,
)

# 加个标注：1978 改革开放 / Annotate: reform & opening up
fig.add_vline(x=1978, line_dash="dot", line_color="gray")
fig.add_annotation(x=1978, y=62, text="1978: Reform & Opening", showarrow=False,
                   yshift=20, font=dict(color="gray"))

fig.show()


**双坐标轴**（左寿命、右 GDP）+ **垂直参考线** + **标注**——`go` 可以做的精细操作 `px` 给不了。
Dual y-axis + vertical reference line + annotation — `go` lets you express details `px` can't.


<a id="13"></a>
## 13. 保存为 HTML / Saving as HTML

Plotly 的杀手用法：导出**独立的 HTML 文件**，发给同事直接在浏览器里看，**完整保留所有交互**。
Killer feature: export to a **standalone HTML file** that anyone can open in a browser **with full interactivity preserved**.


In [ ]:
# 把上面那张著名的 Gapminder 动画保存成 HTML
fig = px.scatter(
    gap, x="gdpPercap", y="lifeExp",
    color="continent", size="pop", size_max=60,
    hover_name="country", log_x=True,
    range_x=[100, 100_000], range_y=[25, 90],
    animation_frame="year", animation_group="country",
    title="Gapminder 1952–2007",
)

out_html = "/tmp/gapminder_animation.html"
fig.write_html(out_html)

import os
print(f"Saved: {out_html}  ({os.path.getsize(out_html)/1024:.0f} KB)")
print("→ 直接双击在浏览器打开就能玩 / Double-click in browser to interact")


**HTML 文件大小一般 3–5 MB**（plotly.js 内嵌）。要省体积可以加 `include_plotlyjs="cdn"`：
File is typically 3-5 MB (plotly.js bundled in). Use `include_plotlyjs="cdn"` to slim it down (loads plotly.js from CDN at view time).

```python
fig.write_html(out_html, include_plotlyjs="cdn")     # 只有几十 KB / few dozen KB
```


<a id="14"></a>
## 14. 从图到 Dashboard / From Plot to Dashboard

单张 Plotly 图已经很好。但**真正的 DS 价值在 Dashboard**——把多张图 + 控件（下拉、滑块、按钮）组合成完整应用。
A single Plotly figure is nice. **The real DS value is in dashboards** — multiple charts + controls (dropdowns, sliders, buttons).

两套主流工具：
Two mainstream tools:

| 工具 / Tool | 写法 / Style | 何时用 / When |
|---|---|---|
| **Streamlit** | 把 Python 脚本变 Web 应用，**最简单** | 数据科学家自己快速做 prototype |
| **Dash** (plotly 官方) | callback-based，**最灵活** | 企业级 dashboard、复杂交互 |

后面 Part 22 (MLOps & Deployment) 我们会专门讲 Streamlit。
We'll cover Streamlit in Part 22 (MLOps & Deployment).

> 💡 **面试常考 / Common interview Q**: "你怎么把模型预测结果给非技术同事看？" 答："Streamlit + Plotly 几小时搞定一个 dashboard，比写 PPT 信息量高十倍。"
> Interview Q: "How would you show model predictions to non-technical colleagues?" A: "Streamlit + Plotly dashboard in a few hours — far more informative than slides."


<a id="15"></a>
## 15. 实战：Gapminder 完整探索 / Hands-on

把本节所有工具串起来做一份 Gapminder 报告。
End-to-end exploration of Gapminder.

**问题 / Questions**:
1. 各大洲发展轨迹有何不同？/ How did continents develop differently?
2. 哪些国家"逆袭"了？/ Which countries had outlier growth?
3. 寿命和 GDP 是什么关系？是线性？非线性？/ life-expectancy vs GDP — what's the shape?


In [ ]:
# Q1: 各大洲发展轨迹 / Continent trajectories
# 按大洲 × 年份算（人口加权）平均寿命
cont_year = (
    gap.assign(life_pop=lambda d: d["lifeExp"] * d["pop"])
       .groupby(["continent", "year"], as_index=False)
       .agg(life_pop=("life_pop", "sum"), pop=("pop", "sum"))
       .assign(lifeExp_w=lambda d: d["life_pop"] / d["pop"])
)

fig = px.line(
    cont_year, x="year", y="lifeExp_w", color="continent",
    markers=True,
    title="Population-weighted mean life expectancy by continent",
    labels={"lifeExp_w": "Life Expectancy (years)"},
    height=450,
)
fig.show()


In [ ]:
# Q2: "逆袭"国家 —— 寿命增长最多的 10 个
# Q2: Biggest life-expectancy gainers
delta = (
    gap.pivot_table(index="country", columns="year", values="lifeExp")
       .assign(delta=lambda d: d[2007] - d[1952])
       .sort_values("delta", ascending=False)
       .head(10)
       .reset_index()
       [["country", 1952, 2007, "delta"]]
)
print(delta.round(1))

fig = px.bar(
    delta.sort_values("delta"),
    y="country", x="delta",
    orientation="h",
    title="Top 10 life-expectancy gainers 1952 → 2007 (years)",
    text="delta",
    color="delta", color_continuous_scale="Greens",
    height=450,
)
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.show()


In [ ]:
# Q3: 寿命 vs GDP 的形状 —— 在 log(GDP) 下接近线性
# Q3: Shape of life-exp vs GDP — near-linear under log(GDP)
fig = px.scatter(
    gap_2007, x="gdpPercap", y="lifeExp",
    log_x=True,
    trendline="ols",                  # 回归线 / OLS line
    trendline_color_override="red",
    color="continent",
    hover_name="country",
    title="lifeExp vs log(gdpPercap), 2007 — near-linear",
    height=500,
)
fig.show()

# 数值确认 / Numerical confirmation
import statsmodels.api as sm
X = sm.add_constant(np.log(gap_2007["gdpPercap"]))
y = gap_2007["lifeExp"]
res = sm.OLS(y, X).fit()
print(f"\nFitted: lifeExp ≈ {res.params['const']:.1f} + "
      f"{res.params['gdpPercap']:.2f} * log(gdpPercap)")
print(f"R² = {res.rsquared:.3f}")


### 📊 Gapminder 结论 / Takeaways

1. **欧洲、北美**早就高位平台化（80+ 岁）；**亚洲、非洲**追赶最快。
   Europe & N. America plateaued early; Asia and Africa caught up fastest.
2. **逆袭榜**前几名都是 1950s 极低起点的国家：阿曼、越南、沙特、印度尼西亚——预期寿命**翻倍**。
   Top gainers all started from extreme lows (Oman, Vietnam, Saudi, Indonesia) — life-expectancy roughly **doubled**.
3. **`lifeExp ≈ a + b·log(GDP)`** 关系**非常稳健**（$R^2 \approx 0.65$）——这意味着：经济翻一倍（log 加 0.69）寿命大约长 **4–6 岁**。**收益递减**，符合直觉。
   The log-linear fit is remarkably robust — doubling income ≈ **+4-6 years** of life-expectancy. Diminishing returns, as expected.

### 💡 业务/政策角度的解读 / Policy interpretation

- 援助贫困国家提升 GDP，**寿命增长的边际效益最大**（左侧斜率最陡）
- 已经在 30K+ GDP 的国家，提升寿命要靠**医疗 / 生活方式**而非 GDP
- Marginal returns of GDP on life expectancy are highest at the low end — aid dollars go furthest there.


<a id="16"></a>
## 16. 小结 / Summary

| 主题 / Topic | 关键 / Key takeaway |
|---|---|
| Two-level API | 95% 用 `px`，精细控制下沉到 `go` |
| Quick start | `px.scatter(df, x=..., y=..., color=..., size=...)` 一行搞定 |
| Hover | `hover_data={...}` 改格式；`hovertemplate=...` 自由模板 |
| 主题 | `pio.templates.default = "plotly_white"` |
| Animation | `animation_frame=` + `animation_group=` —— Plotly 杀手锏 ⭐ |
| Maps | `px.choropleth(..., locations="iso_alpha", color=...)` |
| Subplots | `make_subplots(rows, cols, specs=[...])` |
| 双坐标轴 | `go.Figure()` + `yaxis2=dict(overlaying="y")` |
| 保存 | `fig.write_html(path, include_plotlyjs="cdn")` |
| Dashboard | Streamlit (简单) / Dash (灵活) |

### Plotly vs Matplotlib 决策表 / Choose your tool

| 场景 / Scenario | 推荐 / Pick |
|---|---|
| 论文 / 静态 PDF | matplotlib + seaborn |
| Slack / 博客静态图 | matplotlib + seaborn |
| Stakeholder Demo | **Plotly** |
| Dashboard | **Plotly** + Streamlit |
| 数据量 > 50K 散点 | matplotlib (Plotly 卡) |
| 时间维度的动态变化 | **Plotly animation** ⭐ |
| 地图 | **Plotly choropleth** |
| 论文里的相关性矩阵 | matplotlib heatmap |
| 给非技术老板看 | **Plotly**（他能自己悬停看数）|

### 💡 工业速查 / Industry cheat sheet

```python
# 散点 + 颜色 + 大小 + 对数
px.scatter(df, x=..., y=..., color=..., size=..., log_x=True, hover_name=...)

# 折线多组对比
px.line(df, x="ts", y="metric", color="segment", markers=True)

# 直方图 + 边缘箱线
px.histogram(df, x="x", color="g", marginal="box")

# 动画
px.scatter(df, ..., animation_frame="year", animation_group="id")

# 地图
px.choropleth(df, locations="iso_alpha", color="metric",
              color_continuous_scale="Viridis", projection="natural earth")

# 保存
fig.write_html("plot.html", include_plotlyjs="cdn")
```

### 下一节预告 / Next up

**Part 0.7 · 线性代数** —— 数据科学的真正"硬骨头"。从向量、矩阵到特征值、SVD，每个概念都用 NumPy 复现一遍。看完后 PCA / 推荐系统 / 神经网络都不再神秘。
**Part 0.7 · Linear Algebra** — the hard core of DS. Vectors, matrices, eigen-decomp, SVD — implement each with NumPy. After this, PCA / recommenders / neural nets won't be mysterious.
